# 11h — Net-zero tracking-error optimization

The same 8-asset toy universes from `11a`/`11b`, now re-optimized at a
sequence of CI-reduction *targets* that trace out a CTB-style
compounding glide path ($R_{-}=30\%$ initial cut, $\Delta R=7\%$/year)
rather than an arbitrary grid. The equity case reuses `11a`'s
`min_te_portfolio` QP; the bond case reuses `11b`'s LP-reformulation
approach, extended with a duration-neutrality constraint. Two further
sections add a green-intensity floor and a carbon-momentum ceiling on
top of the CI-reduction target, for both the equity and bond cases.


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, linprog


def _prune_redundant_rows(A, b, tol=1e-10):
    keep, rows = [], []
    for i in range(A.shape[0]):
        cand = np.vstack(rows + [A[i]]) if rows else A[i:i + 1]
        if np.linalg.matrix_rank(cand, tol=tol) > len(rows):
            rows.append(A[i])
            keep.append(i)
    return A[keep], b[keep]


def min_te_portfolio(b, Sigma, A_eq, b_eq, A_ub=None, b_ub=None, lb=None, ub=None):
    # gamma=0 case of the tracking-error minimization from 11a.
    n = len(b)
    lb = np.zeros(n) if lb is None else np.asarray(lb)
    ub = np.ones(n) if ub is None else np.asarray(ub)
    A_eq, b_eq = _prune_redundant_rows(np.asarray(A_eq, dtype=float), np.asarray(b_eq, dtype=float))

    def obj(w):
        d = w - b
        return d @ Sigma @ d

    def obj_grad(w):
        return 2 * Sigma @ (w - b)

    constraints = [{"type": "eq", "fun": lambda w, A=A_eq, c=b_eq: A @ w - c}]
    if A_ub is not None:
        constraints.append({"type": "ineq", "fun": lambda w, A=A_ub, c=b_ub: c - A @ w})

    res = minimize(obj, b.copy(), jac=obj_grad, method="SLSQP",
                    bounds=list(zip(lb, ub)), constraints=constraints,
                    options={"maxiter": 1000, "ftol": 1e-14})
    w = res.x
    w = w * (np.abs(w) >= 1e-10)
    sigma_x = np.sqrt(max((w - b) @ Sigma @ (w - b), 0))
    return w, sigma_x


def glide_path(R_minus, Delta_R, t0, offsets, prepend_zero=True):
    # R(t) = 1 - (1-R_minus)*(1-Delta_R)^(t-t0), a CTB-style compounding
    # reduction target -- the same closed form as 11f/11g, now used as the
    # sequence of *targets* an optimizer re-solves against.
    t = t0 + np.asarray(offsets)
    R = 1 - (1 - R_minus) * (1 - Delta_R) ** (t - t0)
    if prepend_zero:
        R = np.concatenate([[0.0], R])
    return R


def solve_bond_l1_duration_neutral(b, CI, MD, DTS, Sector, R_target, varphi_DTS,
                                    extra_C=None, extra_D=None, ub_mask=None, x0=None):
    # LP reformulation of the L1 AS+DTS bond objective (varphi_AS=1 fixed,
    # no MD penalty term) subject to: budget, duration-neutrality (MD(w)=
    # MD(b)), a CI-reduction target, and optional extra inequality rows
    # (a carbon-momentum ceiling / green-intensity floor, added below).
    b = np.asarray(b, dtype=float)
    CI, MD, DTS = np.asarray(CI, dtype=float), np.asarray(MD, dtype=float), np.asarray(DTS, dtype=float)
    Sector = np.asarray(Sector)
    n = len(b)
    sectors = np.unique(Sector)
    nS = len(sectors)
    s_ji = (Sector[None, :] == sectors[:, None]).astype(float)
    DTS_star = (s_ji * DTS) @ b
    CI_b = CI @ b
    MD_b = b @ MD

    c = np.concatenate([np.zeros(n), 0.5 * np.ones(n), varphi_DTS * np.ones(nS)])
    A_eq = np.vstack([np.concatenate([np.ones(n), np.zeros(n + nS)]),
                       np.concatenate([MD, np.zeros(n + nS)])])
    b_eq = np.array([1.0, MD_b])

    I_n, I_nS = np.eye(n), np.eye(nS)
    Z1, Z2 = np.zeros((n, nS)), np.zeros((nS, n))
    C_DTS = s_ji * DTS
    A_ub = np.block([
        [I_n, -I_n, Z1],
        [-I_n, -I_n, Z1],
        [C_DTS, Z2, -I_nS],
        [-C_DTS, Z2, -I_nS],
    ])
    b_ub = np.concatenate([b, -b, DTS_star, -DTS_star])
    A_ub = np.vstack([A_ub, np.concatenate([CI, np.zeros(n + nS)])])
    b_ub = np.concatenate([b_ub, [(1 - R_target) * CI_b]])
    if extra_C is not None:
        extra_C = np.atleast_2d(extra_C)
        A_ub = np.vstack([A_ub, np.hstack([extra_C, np.zeros((extra_C.shape[0], n + nS))])])
        b_ub = np.concatenate([b_ub, np.atleast_1d(extra_D)])

    ub_w = np.ones(n) if ub_mask is None else np.asarray(ub_mask, dtype=float)
    bounds = list(zip(np.zeros(n), ub_w)) + [(0, 1e5)] * (n + nS)

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        return None
    return res.x[:n]


def solve_bond_l1_duration_neutral_smooth(b, CI, MD, DTS, Sector, R_target, varphi_DTS, x0=None):
    # Direct nonsmooth SLSQP minimize of the same objective, used to
    # cross-check against the LP reformulation above.
    b = np.asarray(b, dtype=float)
    CI, MD, DTS = np.asarray(CI, dtype=float), np.asarray(MD, dtype=float), np.asarray(DTS, dtype=float)
    Sector = np.asarray(Sector)
    n = len(b)
    sectors = np.unique(Sector)
    MD_b = b @ MD
    CI_b = CI @ b

    def R_AS(w):
        return 0.5 * np.abs(w - b).sum()

    def R_DTS(w):
        return np.abs(np.array([((Sector == s) * (w - b) * DTS).sum() for s in sectors])).sum()

    def R_Mix(w):
        return varphi_DTS * R_DTS(w) + R_AS(w)

    cons = [
        {"type": "eq", "fun": lambda w: w.sum() - 1.0},
        {"type": "eq", "fun": lambda w: w @ MD - MD_b},
        {"type": "ineq", "fun": lambda w: (1 - R_target) * CI_b - CI @ w},
    ]
    x0 = b.copy() if x0 is None else x0
    res = minimize(R_Mix, x0, method="SLSQP", bounds=[(0, 1)] * n,
                    constraints=cons, options={"maxiter": 1000, "ftol": 1e-15})
    return res.x


## 1. Equity glide path

The `11a`/`11d` 8-asset equity universe, re-optimized at 8 points along a
$R_-=30\%$, $\Delta R=7\%$/year compounding glide path (an unconstrained
$R=0$ starting point, then $t=2020,\dots,2025,2030$).


In [2]:
b = np.array([0.20, 0.19, 0.17, 0.13, 0.12, 0.08, 0.06, 0.05])
CI = np.array([100.5, 97.2, 250.4, 352.3, 27.1, 54.2, 78.6, 426.7])
beta = np.array([0.30, 1.80, 0.85, 0.83, 1.47, 0.94, 1.67, 1.08])
sigma_tilde = np.array([0.10, 0.05, 0.06, 0.12, 0.15, 0.04, 0.08, 0.07])
sigma_m = 0.18
Sigma = np.outer(beta, beta) * sigma_m ** 2 + np.diag(sigma_tilde ** 2)
n = len(b)
CI_b = b @ CI

R_glide = glide_path(0.30, 0.07, 2020, [0, 1, 2, 3, 4, 5, 10])
w4 = np.zeros((n, len(R_glide)))
sigma_w4 = np.zeros(len(R_glide))
for i, R in enumerate(R_glide):
    w4[:, i], sigma_w4[i] = min_te_portfolio(b, Sigma, np.ones((1, n)), np.array([1.0]),
                                              CI.reshape(1, -1), np.array([(1 - R) * CI_b]))
CI_w4 = w4.T @ CI
R_w4 = 1 - CI_w4 / CI_b
table4 = pd.DataFrame(100 * w4, index=[f"Asset {i+1}" for i in range(n)],
                       columns=[f"R={100*r:.1f}%" for r in R_glide]).T
table4["sigma(w|b), bps"] = 1e4 * sigma_w4
table4["Reduction %"] = 100 * R_w4
display(table4.round(2))

,Asset 1,Asset 2,Asset 3,Asset 4,Asset 5,Asset 6,Asset 7,Asset 8,"sigma(w|b), bps",Reduction %
R=0.0%,20.00,19.00,17.00,13.00,12.00,8.00,6.00,5.0,0.00,0.00
R=30.0%,21.86,18.70,8.06,8.74,13.07,22.57,7.00,0.0,104.10,30.00
R=34.9%,22.21,18.41,5.69,7.66,13.29,25.59,7.15,0.0,126.22,34.90
R=39.5%,22.54,18.15,3.48,6.65,13.51,28.39,7.29,0.0,147.14,39.46
R=43.7%,22.84,17.90,1.43,5.72,13.70,31.00,7.42,0.0,166.79,43.70
R=47.6%,23.02,17.58,0.00,4.56,13.91,33.39,7.53,0.0,185.24,47.64
R=51.3%,22.92,17.04,0.00,2.70,14.18,35.54,7.63,0.0,203.51,51.30
R=66.1%,8.81,0.00,0.00,0.00,21.22,62.31,7.66,0.0,352.42,66.12


## 2. Bond glide path, duration-neutral

This section uses its own 8-asset bond universe -- the same `b`/`CI`
weights and intensities as the equity case, paired with new
`MD`/`DTS`/`Sector` bond fields, rather than the smaller 4-asset universe
from `11b`. The L1 active-share+DTS objective is minimized subject to
duration-neutrality ($MD(w)=MD(b)$) and the same glide-path CI target,
solved both via LP reformulation and directly via SLSQP, then
cross-checked against each other.


In [3]:
b5 = np.array([0.20, 0.19, 0.17, 0.13, 0.12, 0.08, 0.06, 0.05])
CI5 = np.array([100.5, 97.2, 250.4, 352.3, 27.1, 54.2, 78.6, 426.7])
MD5 = np.array([3.10, 8.60, 7.20, 5.00, 4.70, 2.10, 8.10, 2.60])
DTS5 = np.array([100, 155, 575, 436, 159, 145, 804, 365])
Sector5 = np.array([1, 2, 1, 1, 2, 2, 2, 1])
varphi5 = 0.005

rows5 = []
w_prev = b5.copy()
for R in R_glide:
    w_lp = solve_bond_l1_duration_neutral(b5, CI5, MD5, DTS5, Sector5, R, varphi5, x0=w_prev)
    w_sm = solve_bond_l1_duration_neutral_smooth(b5, CI5, MD5, DTS5, Sector5, R, varphi5, x0=w_prev)
    if w_lp is not None:
        w_prev = w_lp
    rows5.append({"R target": R, "max|w_lp - w_fmincon|": np.max(np.abs(w_lp - w_sm)) if w_lp is not None else np.nan,
                   "CI(w_lp)": w_lp @ CI5 if w_lp is not None else np.nan,
                   "MD(w_lp)": w_lp @ MD5 if w_lp is not None else np.nan, "MD(b)": b5 @ MD5,
                   **{f"w_lp[{i+1}]": (100 * w_lp[i] if w_lp is not None else np.nan) for i in range(8)}})
display(pd.DataFrame(rows5).round(4))

,R target,max|w_lp - w_fmincon|,CI(w_lp),MD(w_lp),MD(b),w_lp[1],w_lp[2],w_lp[3],w_lp[4],w_lp[5],w_lp[6],w_lp[7],w_lp[8]
0,0.0000,0.0000,160.5740,5.476,5.476,20.0000,19.0000,17.0000,13.0,12.0000,8.0000,6.0000,5.0
1,0.3000,0.0000,112.4018,5.476,5.476,20.0000,13.9856,25.4324,0.0,28.9716,8.0000,3.6104,0.0
2,0.3490,0.0000,104.5337,5.476,5.476,20.0000,17.7905,20.9630,0.0,30.7143,8.0000,2.5322,0.0
3,0.3946,0.0001,97.2163,5.476,5.476,20.0000,19.0000,17.7799,0.0,35.8443,5.6712,1.7045,0.0
4,0.4370,0.0000,90.4112,5.476,5.476,13.9760,19.0000,17.0000,0.0,43.5223,6.4573,0.0443,0.0
5,0.4764,0.0000,84.0824,5.476,5.476,17.6400,19.0000,13.6439,0.0,48.7984,0.9176,0.0000,0.0
6,0.5130,0.0000,78.1966,5.476,5.476,16.0192,19.0000,11.6523,0.0,53.3285,0.0000,0.0000,0.0
7,0.6612,0.0000,54.4005,5.476,5.476,5.0183,19.0000,4.6117,0.0,71.3699,0.0000,0.0000,0.0


## 3. Equity glide path with green-intensity floor and carbon-momentum ceiling

The equity universe from the first section, extended with a
**green-intensity floor** ($GI(w)\ge2\times GI(b)$) and a
**carbon-momentum ceiling** ($CM(w)\le-3\%$), plus a hard exclusion of
any asset whose own momentum exceeds $5\%$ (via the upper bound). The
$R=0$ point is left at the benchmark itself, unsolved.

**Note**: at the two highest glide-path targets ($R=51.3\%$, $R=66.1\%$)
the CI-reduction target is no longer jointly feasible together with the
CM ceiling, GI floor, and asset-level momentum exclusion -- the maximum
CI reduction achievable under those overlay constraints in this 8-asset
toy universe is $\approx51\%$ (confirmed independently via a feasibility
LP). SLSQP settles at the closest feasible point rather than erroring,
so `Reduction %` plateaus around 51% instead of continuing to track the
glide path, since the solver's exit status isn't checked at these
points.


In [4]:
GI = np.array([10.2, 45.3, 7.5, 0.0, 0.0, 35.6, 17.8, 3.0]) / 100
CM = np.array([-3.1, -1.2, -5.8, -1.4, 7.4, -2.6, 1.2, -8.0]) / 100
CM_star, CM_plus, greenness = -0.03, 0.05, 1
CM_b, GI_b = b @ CM, b @ GI

w6 = np.zeros((n, len(R_glide)))
sigma_w6 = np.zeros(len(R_glide))
w6[:, 0] = b
ub6 = (CM <= CM_plus).astype(float)
for i in range(1, len(R_glide)):
    C = np.vstack([CI, CM, -GI])
    D = np.array([(1 - R_glide[i]) * CI_b, CM_star, -(1 + greenness) * GI_b])
    w6[:, i], sigma_w6[i] = min_te_portfolio(b, Sigma, np.ones((1, n)), np.array([1.0]), C, D, ub=ub6)

CI_w6, CM_w6, GI_w6 = w6.T @ CI, w6.T @ CM, w6.T @ GI
R_w6 = 1 - CI_w6 / CI_b
table6 = pd.DataFrame(100 * w6, index=[f"Asset {i+1}" for i in range(n)],
                       columns=[f"R={100*r:.1f}%" for r in R_glide]).T
table6["sigma(w|b), bps"] = 1e4 * sigma_w6
table6["Reduction %"] = 100 * R_w6
table6["CM(w) %"] = 100 * CM_w6
table6["GI(w) %"] = 100 * GI_w6
display(table6.round(2))

,Asset 1,Asset 2,Asset 3,Asset 4,Asset 5,Asset 6,Asset 7,Asset 8,"sigma(w|b), bps",Reduction %,CM(w) %,GI(w) %
R=0.0%,20.00,19.00,17.00,13.0,12.0,8.00,6.0,5.00,0.00,0.00,-1.66,15.99
R=30.0%,5.26,20.96,3.35,0.0,0.0,60.06,0.0,10.37,370.16,30.96,-3.00,31.98
R=34.9%,3.51,17.27,7.27,0.0,0.0,64.69,0.0,7.25,376.38,34.90,-3.00,31.98
R=39.5%,1.49,13.00,11.82,0.0,0.0,70.05,0.0,3.64,398.30,39.46,-3.00,31.98
R=43.7%,0.00,8.82,15.02,0.0,0.0,75.37,0.0,0.79,430.94,43.70,-3.00,31.98
R=47.6%,0.02,4.16,14.32,0.0,0.0,81.51,0.0,0.00,472.44,47.64,-3.00,31.98
R=51.3%,0.00,0.00,12.19,0.0,0.0,88.19,0.0,0.00,515.52,51.22,-3.00,32.31
R=66.1%,0.00,0.00,12.53,0.0,0.0,87.44,0.0,0.00,516.96,50.95,-3.00,32.07


## 4. Bond glide path with the same extensions

The bond universe from the previous section, extended with the same
green-intensity floor and carbon-momentum ceiling, LP-solved only.
**Note**: this glide path does *not* prepend the unconstrained $R=0$
point -- all 7 targets already include the $30\%$ initial cut.

As with the equity case above, the two highest targets ($R=51.3\%$,
$R=66.1\%$) are infeasible jointly with the CM ceiling, GI floor, and
momentum exclusion in this 8-asset universe (confirmed via a standalone
feasibility LP) -- `solve_bond_l1_duration_neutral` returns `None` and
those rows show `NaN` when the LP has no feasible solution.


In [5]:
# Same 8-asset bond universe as the duration-neutral case above
# (b/CI/MD/DTS/Sector), extended with the same GI/CM overlay as the
# equity case.
GI7 = np.array([10.2, 45.3, 7.5, 0.0, 0.0, 35.6, 17.8, 3.0]) / 100
CM7 = np.array([-3.1, -1.2, -5.8, -1.4, 7.4, -2.6, 1.2, -8.0]) / 100
b7 = np.array([0.20, 0.19, 0.17, 0.13, 0.12, 0.08, 0.06, 0.05])
CI7 = np.array([100.5, 97.2, 250.4, 352.3, 27.1, 54.2, 78.6, 426.7])
MD7 = np.array([3.10, 8.60, 7.20, 5.00, 4.70, 2.10, 8.10, 2.60])
DTS7 = np.array([100, 155, 575, 436, 159, 145, 804, 365])
Sector7 = np.array([1, 2, 1, 1, 2, 2, 2, 1])
varphi7 = 0.005
CM_star7, CM_plus7, greenness7 = -0.02, 0.05, 1
CI7_b, CM7_b, GI7_b = b7 @ CI7, b7 @ CM7, b7 @ GI7

R_glide7 = glide_path(0.30, 0.07, 2020, [0, 1, 2, 3, 4, 5, 10], prepend_zero=False)
ub7 = (CM7 <= CM_plus7).astype(float)

rows7 = []
w_prev7 = b7.copy()
for R in R_glide7:
    extra_C = np.vstack([CM7, -GI7])
    extra_D = np.array([CM_star7, -(1 + greenness7) * GI7_b])
    w_lp7 = solve_bond_l1_duration_neutral(b7, CI7, MD7, DTS7, Sector7, R, varphi7,
                                            extra_C=extra_C, extra_D=extra_D, ub_mask=ub7, x0=w_prev7)
    if w_lp7 is not None:
        w_prev7 = w_lp7
        rows7.append({"R target": R, "CI(w)": w_lp7 @ CI7, "MD(w)": w_lp7 @ MD7, "MD(b)": b7 @ MD7,
                       "CM(w) %": 100 * (w_lp7 @ CM7), "GI(w) %": 100 * (w_lp7 @ GI7),
                       **{f"w[{i+1}]": 100 * w_lp7[i] for i in range(8)}})
    else:
        rows7.append({"R target": R, "CI(w)": np.nan})
display(pd.DataFrame(rows7).round(2))

,R target,CI(w),MD(w),MD(b),CM(w) %,GI(w) %,w[1],w[2],w[3],w[4],w[5],w[6],w[7],w[8]
0,0.30,112.40,5.48,5.48,-2.81,31.98,4.28,34.78,21.03,0.0,0.0,39.91,0.0,0.0
1,0.35,104.53,5.48,5.48,-2.57,31.98,13.80,38.94,13.86,0.0,0.0,33.40,0.0,0.0
2,0.39,97.22,5.48,5.48,-2.35,32.37,20.48,42.72,7.73,0.0,0.0,29.07,0.0,0.0
3,0.44,90.41,5.48,5.48,-2.15,32.80,26.34,46.23,2.11,0.0,0.0,25.32,0.0,0.0
4,0.48,84.08,5.48,5.48,-2.01,35.52,19.02,49.01,0.00,0.0,0.0,31.97,0.0,0.0
5,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
